# Part 3 – NLP and Sequence Modeling Mini Project
**Dataset:** Customer Support Text Classification  
**Goal:** Build an NLP pipeline and compare traditional vectorisation with sequence-based deep-learning ideas.

---

## Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

print('All imports successful.')

---
## Task 1 – Dataset Understanding

In [ ]:
df = pd.read_csv('customer_support_text_classification.csv')
print(f'Shape            : {df.shape}')
print(f'Columns          : {df.columns.tolist()}')
print(f'Null values      :\n{df.isnull().sum()}')
df.head()

In [ ]:
# ── Class distribution ────────────────────────────────────────────────────────
print('=== Target Label Distribution ===')
print(df['sentiment_label'].value_counts())

# ── Channel distribution ─────────────────────────────────────────────────────
print('\n=== Channel Distribution ===')
print(df['channel'].value_counts())

# ── Word-count stats ─────────────────────────────────────────────────────────
print('\n=== Word Count Statistics ===')
print(df['word_count'].describe())

In [ ]:
# ── Sample records per class ─────────────────────────────────────────────────
print('=== Sample Customer Messages ===')
for label in ['positive', 'neutral', 'negative']:
    sample = df[df['sentiment_label'] == label]['customer_message'].iloc[0]
    print(f'[{label.upper():8s}] {sample}')
    print()

In [ ]:
# ── Visualisations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Dataset Overview', fontsize=14, fontweight='bold')

# Class distribution
counts = df['sentiment_label'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=['#e74c3c', '#95a5a6', '#2ecc71'])
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Word count distribution
df['word_count'].plot(kind='hist', bins=20, ax=axes[1],
                      color='steelblue', edgecolor='white')
axes[1].axvline(df['word_count'].mean(), color='red',
                linestyle='--', label=f'Mean = {df["word_count"].mean():.1f}')
axes[1].set_title('Word Count Distribution')
axes[1].set_xlabel('Word Count'); axes[1].legend()

# Channel breakdown
ch = df['channel'].value_counts()
axes[2].pie(ch.values, labels=ch.index, autopct='%1.1f%%', startangle=90)
axes[2].set_title('Channel Distribution')

plt.tight_layout()
plt.show()

**Dataset Summary**

| Property | Value |
|---|---|
| Total records | 1 500 |
| Target classes | `positive`, `neutral`, `negative` |
| Class balance | Roughly balanced (~33 % each) |
| Average word count | 12.7 words |
| Channels | email, social, phone, chat, app |
| Missing values | None |

> **Note:** The dataset is synthetic. Many messages are repeated across records within the same class, which leads to very high model accuracy (essentially memorisation). In a real-world scenario, messages would be more diverse.

---

## Task 2 – Text Preprocessing

In [ ]:
# Custom English stopword list (no external dependencies)
STOPWORDS = set("""
a about above after again against all am an and any are aren't as at be
because been before being below between both but by can't cannot could
couldn't did didn't do does doesn't doing don't down during each few for
from further get got hadn't has hasn't have haven't having he'd he'll he's
hence her here here's hers herself him himself his how how's however i i'd
i'll i'm i've if in into is isn't it it's its itself let's me more most
mustn't my myself no nor not of off on once only or other ought our ours
ourselves out over own same shan't she she'd she'll she's should shouldn't
so some such than that that's the their theirs them themselves then there
there's therefore these they they'd they'll they're they've this those
through to too under until up very was wasn't we we'd we'll we're we've
were weren't what what's when when's where where's which while who who's
whom why why's will with won't would wouldn't you you'd you'll you're you've
your yours yourself yourselves
""".split())

def preprocess(text: str) -> str:
    """Full preprocessing pipeline for a single message."""
    # 1. Lowercase
    text = str(text).lower()
    # 2. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # 3. Remove digits
    text = re.sub(r'\d+', '', text)
    # 4. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 5. Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 6. Tokenise and remove stopwords + short tokens
    tokens = [t for t in text.split()
              if t not in STOPWORDS and len(t) > 1]
    return ' '.join(tokens)

df['cleaned_message'] = df['customer_message'].apply(preprocess)

# Show before / after
print('Before / After preprocessing examples:\n')
for _, row in df.sample(4, random_state=42).iterrows():
    print(f'  Original : {row["customer_message"]}')
    print(f'  Cleaned  : {row["cleaned_message"]}')
    print()

**Preprocessing steps applied:**
1. **Lowercasing** – ensures `Refund` and `refund` are treated identically.
2. **URL removal** – ticket messages may contain links irrelevant to sentiment.
3. **Digit removal** – ticket numbers (e.g. `78732`) are noise for sentiment.
4. **Punctuation removal** – `!` and `.` don't carry consistent semantic meaning across messages.
5. **Stopword removal** – function words (`the`, `is`, `I`) add noise without sentiment signal.
6. **Short-token filtering** – removes single-character artefacts.

---

## Task 3 – Text Vectorisation

In [ ]:
X = df['cleaned_message']
y = df['sentiment_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

In [ ]:
# ── Bag of Words ──────────────────────────────────────────────────────────────
bow_vec = CountVectorizer(max_features=5000)
X_train_bow = bow_vec.fit_transform(X_train)
X_test_bow  = bow_vec.transform(X_test)

print(f'BoW  – vocabulary size : {len(bow_vec.vocabulary_)}')
print(f'BoW  – train matrix    : {X_train_bow.shape}')

# Show top tokens
freqs = np.asarray(X_train_bow.sum(axis=0)).flatten()
top10 = sorted(zip(bow_vec.get_feature_names_out(), freqs),
               key=lambda x: x[1], reverse=True)[:10]
print(f'\nTop 10 tokens by frequency: {[w for w, _ in top10]}')

In [ ]:
# ── TF-IDF (unigrams + bigrams) ───────────────────────────────────────────────
tfidf_vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf_vec.fit_transform(X_train)
X_test_tfidf  = tfidf_vec.transform(X_test)

print(f'TF-IDF – vocabulary size : {len(tfidf_vec.vocabulary_)}')
print(f'TF-IDF – train matrix    : {X_train_tfidf.shape}')

In [ ]:
# ── Tokeniser-based sequences (manual, no keras) ──────────────────────────────
# Build a vocabulary
all_tokens = [tok for doc in X_train for tok in doc.split()]
vocab = {word: idx+1 for idx, (word, _) in
         enumerate(Counter(all_tokens).most_common(5000))}
vocab['<UNK>'] = 0

MAX_LEN = 20  # pad / truncate to 20 tokens

def text_to_seq(text, vocab, max_len):
    tokens = text.split()[:max_len]
    seq = [vocab.get(t, 0) for t in tokens]
    # Pad
    seq += [0] * (max_len - len(seq))
    return seq

X_train_seq = np.array([text_to_seq(t, vocab, MAX_LEN) for t in X_train])
X_test_seq  = np.array([text_to_seq(t, vocab, MAX_LEN) for t in X_test])

print(f'Vocabulary size   : {len(vocab)}')
print(f'Sequence shape    : {X_train_seq.shape}  (samples × max_len)')
print(f'\nExample sequence  : {X_train_seq[0]}')
print(f'Original message  : {X_train.iloc[0]}')

### Why must text be converted to vectors?

Machine-learning and deep-learning models are fundamentally mathematical functions. They perform operations such as matrix multiplication, dot products, and gradient descent — all of which require numeric inputs. Raw text strings cannot be fed directly into these functions.

| Method | Representation | Pros | Cons |
|---|---|---|---|
| **Bag of Words** | Token count vector | Simple, interpretable | Ignores word order; high dimensionality |
| **TF-IDF** | Weighted frequency vector | Down-weights common words | Still ignores order and context |
| **Tokeniser sequences** | Integer index per token | Preserves order; input to RNN/LSTM | Needs embedding layer to be meaningful |
| **Word Embeddings** | Dense semantic vector | Captures meaning & similarity | Requires pre-training or large data |

---

## Task 4 – Baseline Models

In [ ]:
# ── Model 1: Naive Bayes + Bag of Words ───────────────────────────────────────
nb = MultinomialNB()
nb.fit(X_train_bow, y_train)
y_pred_nb = nb.predict(X_test_bow)

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb  = f1_score(y_test, y_pred_nb, average='weighted')

print('=== Naive Bayes + Bag of Words ===')
print(f'Accuracy   : {acc_nb:.4f}')
print(f'Weighted F1: {f1_nb:.4f}')
print()
print(classification_report(y_test, y_pred_nb))

In [ ]:
# ── Model 2: Logistic Regression + TF-IDF ────────────────────────────────────
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr, average='weighted')

print('=== Logistic Regression + TF-IDF ===')
print(f'Accuracy   : {acc_lr:.4f}')
print(f'Weighted F1: {f1_lr:.4f}')
print()
print(classification_report(y_test, y_pred_lr))

In [ ]:
# ── Evaluation plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Baseline Model Evaluation – Customer Support Sentiment',
             fontsize=14, fontweight='bold')

# Bar chart
model_names = ['Naive Bayes\n(BoW)', 'Logistic Regression\n(TF-IDF)']
accs = [acc_nb, acc_lr]
f1s  = [f1_nb,  f1_lr]
x = np.arange(len(model_names))
w = 0.35
axes[0].bar(x - w/2, accs, w, label='Accuracy',    color='steelblue')
axes[0].bar(x + w/2, f1s,  w, label='Weighted F1', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, fontsize=9)
axes[0].set_ylim(0, 1.05); axes[0].set_title('Accuracy & Weighted F1')
axes[0].legend(); axes[0].set_ylabel('Score')

# Confusion matrices
labels = ['negative', 'neutral', 'positive']
for ax, pred, title, cmap in zip(
        axes[1:],
        [y_pred_nb, y_pred_lr],
        ['Naive Bayes (BoW)', 'Logistic Regression (TF-IDF)'],
        ['Blues', 'Greens']):
    cm = confusion_matrix(y_test, pred, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap=cmap,
                xticklabels=labels, yticklabels=labels)
    ax.set_title(f'Confusion Matrix\n{title}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('results/model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/model_evaluation.png')

**Results Commentary**

Both models achieve near-perfect accuracy on this dataset. This is expected because the dataset is **synthetic** — it contains only ~261 unique messages that are repeated across 1 500 records. Any classifier essentially memorises the limited set of sentence patterns. In production, accuracy would be considerably lower on novel, unseen customer messages.

For a real-world benchmark:
- **Naive Bayes + BoW** is a strong, fast baseline suitable for short texts.
- **Logistic Regression + TF-IDF** generally outperforms Naive Bayes on larger, noisier datasets by better handling feature importance.

---

## Task 5 – Sequence Model Architecture (LSTM)

Full LSTM training requires TensorFlow/Keras or PyTorch, which are not available in this environment. Below we define the **complete architecture design** and explain how each layer processes the input.

### Architecture Overview

```
Input Sequence  → [12, 45, 0, 0, 88, ...]   (integer token IDs, length = MAX_LEN = 20)
       ↓
Embedding Layer → (20, 64)  dense vector per token
       ↓
LSTM Layer      → (64 units, return_sequences=False)
       ↓
Dropout (0.3)
       ↓
Dense (32, ReLU)
       ↓
Output Dense (3, Softmax)   →  [P(negative), P(neutral), P(positive)]
```

### Keras Pseudo-code

In [ ]:
# ── LSTM Architecture (pseudo-code / design) ──────────────────────────────────
# This cell shows what the model would look like with TensorFlow/Keras.
# It does NOT execute (tensorflow not installed), but demonstrates
# the full architectural thinking.

LSTM_ARCHITECTURE = """
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

VOCAB_SIZE  = 5001   # vocabulary size + 1 for <PAD>
EMBED_DIM   = 64     # embedding dimensionality
LSTM_UNITS  = 64     # hidden state size
NUM_CLASSES = 3      # negative / neutral / positive
MAX_LEN     = 20     # padded sequence length

model = Sequential([
    # 1. Embedding Layer
    #    Converts integer token IDs → dense vectors of size EMBED_DIM.
    #    Each token learns a position in a semantic space.
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
              input_length=MAX_LEN),

    # 2. LSTM Layer
    #    Processes the sequence token-by-token, maintaining a hidden state
    #    (short-term memory) and cell state (long-term memory via gates).
    #    return_sequences=False → only the final hidden state is passed on.
    LSTM(units=LSTM_UNITS, return_sequences=False),

    # 3. Dropout (regularisation)
    Dropout(0.3),

    # 4. Dense hidden layer
    Dense(32, activation='relu'),

    # 5. Output layer
    #    Softmax → probability distribution over 3 classes.
    Dense(NUM_CLASSES, activation='softmax'),
])

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',   # integer class labels
    metrics=['accuracy']
)
model.summary()

# Training
history = model.fit(
    X_train_seq, y_train_enc,   # integer-encoded labels
    validation_split=0.1,
    epochs=10,
    batch_size=32
)
"""

print(LSTM_ARCHITECTURE)

In [ ]:
# ── Visualise the LSTM data-flow ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
ax.axis('off')
ax.set_xlim(0, 10); ax.set_ylim(0, 10)

layers = [
    (5, 9.2, 'Input Sequence',        '#dce8f5', '20 integer token IDs'),
    (5, 7.5, 'Embedding Layer',       '#d5e8d4', 'shape (20, 64) — token → dense vector'),
    (5, 5.8, 'LSTM Layer (64 units)', '#ffe6cc', 'hidden & cell state across 20 steps'),
    (5, 4.1, 'Dropout (0.3)',         '#f8cecc', 'regularisation — prevents overfitting'),
    (5, 2.4, 'Dense (32, ReLU)',      '#e1d5e7', 'non-linear feature compression'),
    (5, 0.7, 'Output Dense (3, Softmax)', '#fff2cc',
     '[P(neg), P(neu), P(pos)]'),
]

for (x, y, name, color, desc) in layers:
    rect = plt.Rectangle((x-3.5, y-0.5), 7, 1,
                          facecolor=color, edgecolor='#555', linewidth=1.5,
                          zorder=2)
    ax.add_patch(rect)
    ax.text(x, y+0.05, name, ha='center', va='center',
            fontsize=10, fontweight='bold', zorder=3)
    ax.text(x, y-0.3, desc, ha='center', va='center',
            fontsize=7.5, color='#444', zorder=3)
    # Arrow down
    if y > 0.7:
        ax.annotate('', xy=(x, y-0.5), xytext=(x, y-1.2),
                    arrowprops=dict(arrowstyle='->', color='#333',
                                   lw=1.5), zorder=1)

ax.set_title('LSTM Architecture for Sentiment Classification',
             fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('results/lstm_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/lstm_architecture.png')

### Layer-by-Layer Explanation

| Layer | Role |
|---|---|
| **Input Sequence** | Each message is encoded as a fixed-length (20) integer array, where each integer is a token ID in the vocabulary. Padding (0) is added for shorter messages. |
| **Embedding Layer** | Maps each integer ID to a trainable 64-dimensional dense vector. Semantically similar words learn similar vectors during training. |
| **LSTM Layer** | Processes the sequence one token at a time. The *forget gate* discards irrelevant past context; the *input gate* adds new information; the *output gate* determines what to pass forward. This mechanism gives LSTM its long-term memory. |
| **Dropout** | Randomly zeros 30 % of activations during training to prevent the model from over-relying on any single neuron (regularisation). |
| **Dense (ReLU)** | A fully-connected layer that compresses the LSTM's final hidden state into a smaller representation. |
| **Output (Softmax)** | Produces a probability distribution across 3 classes. The class with the highest probability is the predicted sentiment. |

**Loss function:** `sparse_categorical_crossentropy` — measures how far the predicted probability distribution is from the true one-hot label.  
**Evaluation metric:** Accuracy and Weighted F1-score.

---

## Task 6 – Attention and Transformer Reflection

### 1. Why do RNNs struggle with long-term dependencies?

A vanilla RNN passes a single hidden state from one time step to the next. When a sequence is long, gradients must flow backward through every time step during training. Each multiplication through the recurrent weight matrix causes the gradient to either **shrink exponentially** (vanishing gradient) or **explode**. In practice this means the model cannot learn that a word at position 1 is relevant to a prediction at position 50 — the signal is lost across the many intermediate multiplications.

### 2. How do LSTMs help with memory?

LSTMs introduce a **cell state** — a separate memory lane — alongside the hidden state. Three learnable *gates* control information flow:

- **Forget gate** – decides what fraction of the cell state to erase.
- **Input gate** – decides what new information to write into the cell.
- **Output gate** – decides what part of the cell state to expose as the hidden state.

Because the cell state is updated additively (not multiplicatively across every step), gradients can flow further back in time without vanishing. This makes LSTMs capable of capturing dependencies dozens of tokens apart.

### 3. What does Attention solve in sequence-to-sequence tasks?

In encoder-decoder architectures (e.g. machine translation), the encoder must compress an entire input sequence into a single context vector. For long inputs this vector is a bottleneck — it cannot faithfully represent every relevant word.  

**Attention** allows the decoder to directly query all encoder hidden states at each decoding step, computing a *weighted sum* over them. The weights (attention scores) reflect how relevant each input token is to the current output token. This eliminates the bottleneck: the decoder "attends" to the most relevant parts of the input at each step, enabling much better translations and summaries of long documents.

### 4. Why are Transformers important in modern NLP and Generative AI?

Transformers (*Attention is All You Need*, Vaswani et al., 2017) discard recurrence entirely and rely on **self-attention**, where every token attends to every other token in the sequence simultaneously. Key advantages:

| Property | Benefit |
|---|---|
| **Parallelism** | Unlike RNNs, all positions are processed simultaneously → much faster training on GPUs. |
| **Long-range dependencies** | Any two tokens interact directly in one layer, regardless of distance. |
| **Scalability** | Performance improves predictably with more data and parameters (*scaling laws*). |
| **Transfer learning** | Pre-trained models (BERT, GPT, T5) can be fine-tuned on small task-specific datasets with state-of-the-art results. |

Transformers are the backbone of **every major Generative AI system** today — GPT-4, Claude, Gemini, LLaMA — and have also revolutionised vision (ViT), protein folding (AlphaFold), and multi-modal AI. Their ability to model arbitrary relationships in sequences at scale makes them the dominant architecture for NLP and beyond.

---

## Summary

In [ ]:
# ── Save evaluation CSV ───────────────────────────────────────────────────────
eval_df = pd.DataFrame({
    'model'       : ['Naive Bayes (BoW)', 'Logistic Regression (TF-IDF)'],
    'accuracy'    : [round(acc_nb, 4), round(acc_lr, 4)],
    'weighted_f1' : [round(f1_nb,  4), round(f1_lr,  4)]
})
eval_df.to_csv('results/model_evaluation.csv', index=False)
print('Saved → results/model_evaluation.csv')
print(eval_df.to_string(index=False))

# ── Save sample predictions ───────────────────────────────────────────────────
test_df = df.loc[X_test.index].copy()
test_df['predicted'] = y_pred_lr
sample_preds = test_df[['customer_message', 'sentiment_label', 'predicted']].head(20)

with open('results/sample_predictions.txt', 'w') as f:
    f.write('Sample Predictions – Logistic Regression + TF-IDF\n')
    f.write('=' * 70 + '\n')
    for _, row in sample_preds.iterrows():
        match = '✓' if row['sentiment_label'] == row['predicted'] else '✗'
        f.write(
            f"{match} Actual: {row['sentiment_label']:8s} | "
            f"Predicted: {row['predicted']:8s}\n"
            f"  Message: {row['customer_message'][:80]}\n\n"
        )
print('Saved → results/sample_predictions.txt')

### Project Summary Table

| Task | Approach | Output |
|---|---|---|
| 1 – Dataset Understanding | `pandas` EDA + visualisations | Class distribution, word-count stats, channel breakdown |
| 2 – Preprocessing | Lowercase, remove digits/punct, stopword removal | `cleaned_message` column |
| 3 – Vectorisation | BoW, TF-IDF (1-2 grams), integer sequences | Sparse matrices + padded sequences |
| 4 – Baseline Models | Naive Bayes + BoW; Logistic Regression + TF-IDF | `model_evaluation.png`, `model_evaluation.csv` |
| 5 – Sequence Architecture | LSTM design + layer explanation | `lstm_architecture.png` |
| 6 – Reflection | RNN limits, LSTM gates, attention, transformers | Written explanation in notebook |